# Phase 2 — Negative Review Classification

This notebook answers one question: **among orders that were delivered late, which are likely to receive a negative review (1–2 stars)?**

It is designed to run from top to bottom. The notebook keeps the analysis readable; reusable data, modelling and evaluation logic lives in `src/`.


## 1. Problem and modelling population

**Target**
- `1`: review score 1–2 (negative review)
- `0`: review score 3–5

**Population:** delivered orders that were late and subsequently received a review.

**Prediction point:** immediately after the late delivery is completed, before the customer's review is known.

**Key feature:** Phase 1 defines `days_early` so negative values mean late (for example, `-10` means 10 days late). Late delivery is the population filter; the model predicts the review outcome.

Review IDs, review text and review timestamps are excluded from predictors because they are outcome-side information. Phase 2 reuses the cleaned Phase 1 analytical tables rather than repeating general cleaning.


### Load Phase 1 inputs

The notebook uses `data/processed/order_level.csv` (one row per order) and `item_level.csv` (one row per order item).


In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.base import clone
from sklearn.model_selection import (
    StratifiedKFold,
    cross_validate,
    cross_val_predict,
    train_test_split,
)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.classification_data import (
    CATEGORICAL_FEATURES,
    MODEL_FEATURES,
    NUMERIC_FEATURES,
    TARGET_COLUMN,
    build_classification_dataset,
    review_timing_audit,
    split_features_target,
)
from src.classification_evaluation import (
    best_threshold_by_metric,
    classification_metrics,
    threshold_table,
)
from src.classification_model import (
    build_dummy_pipeline,
    build_logistic_pipeline,
    predict_from_artifact,
    save_inference_artifact,
)

RANDOM_SEED = 42
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
order_level = pd.read_csv(PROCESSED_DIR / "order_level.csv")
item_level = pd.read_csv(PROCESSED_DIR / "item_level.csv")
print(f"order_level: {order_level.shape}; item_level: {item_level.shape}")

### Build and check the modelling dataset

Create one row per eligible late order. Product/item information is aggregated deterministically before modelling; learned imputation, scaling and encoding stay inside the scikit-learn Pipeline.


In [ ]:
classification_data = build_classification_dataset(order_level, item_level)
assert classification_data["order_id"].is_unique

column_audit = pd.DataFrame({
    "dtype": classification_data.dtypes.astype(str),
    "missing_n": classification_data.isna().sum(),
    "missing_pct": classification_data.isna().mean(),
    "unique_n": classification_data.nunique(dropna=False),
})
print(f"classification dataset: {classification_data.shape}")
display(column_audit)
display(classification_data[TARGET_COLUMN].value_counts(dropna=False).rename("n").to_frame())
display(classification_data[NUMERIC_FEATURES].describe().T)
display(review_timing_audit(order_level).to_frame())

### Feature and leakage check

Before modelling, verify that only information available at the prediction point enters the model.

| Information | Use | Why |
| --- | --- | --- |
| Review score | Target only | Defines negative vs non-negative review |
| Review ID, text, timestamps | Excluded | Outcome-side leakage |
| Order/customer/product/seller IDs | Excluded | Trace keys; avoid entity memorisation |
| Order status / late flag | Cohort filter only | Constant within the modelling population |
| Delivery timing incl. `days_early` | Predictor | Known when delivery is completed |
| Item, value, freight, payment | Predictor | Order characteristics |
| Customer/seller geography | Predictor | Coarse geography without entity IDs |
| Purchase month/weekday | Predictor | Seasonal context |
| Product category/size | Predictor | Product characteristics |

The code below prints the exact allow-listed model features used in this run.


In [ ]:
feature_audit = pd.DataFrame({
    "feature": MODEL_FEATURES,
    "type": ["numeric" if name in NUMERIC_FEATURES else "categorical" for name in MODEL_FEATURES],
    "candidate_model_input": True,
    "available_after_delivery": True,
})
display(feature_audit)
assert not any(name.startswith("review_") for name in MODEL_FEATURES)
assert "order_id" not in MODEL_FEATURES

### Protected train/test split

Create the test set once. All model comparison and threshold selection use the training set only; the test set is opened once for final evaluation.


In [ ]:
X, y = split_features_target(classification_data)
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_SEED,
)
print(f"training rows: {len(X_train):,}; protected test rows: {len(X_test):,}")

## 2. Model development

### Baseline and Logistic Regression

A `DummyClassifier` provides the no-skill baseline. We compare two Logistic Regression configurations on the **same five stratified folds**: standard Logistic Regression and class-weighted Logistic Regression. Preprocessing is fitted inside each fold to prevent leakage.


In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
scoring = {
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "balanced_accuracy": "balanced_accuracy",
    "pr_auc": "average_precision",
    "roc_auc": "roc_auc",
}

candidates = {
    "Dummy baseline": build_dummy_pipeline(strategy="prior"),
    "Logistic Regression": build_logistic_pipeline(random_state=RANDOM_SEED),
    "Logistic Regression (balanced)": build_logistic_pipeline(
        random_state=RANDOM_SEED, class_weight="balanced"
    ),
}

cv_rows = []
for name, pipeline in candidates.items():
    scores = cross_validate(pipeline, X_train, y_train, cv=cv, scoring=scoring)
    row = {"model": name}
    for metric in scoring:
        values = scores[f"test_{metric}"]
        row[f"{metric}_mean"] = values.mean()
        row[f"{metric}_sd"] = values.std()
    cv_rows.append(row)

cv_summary = pd.DataFrame(cv_rows).set_index("model")
display(cv_summary.round(3))


### Select the Logistic Regression configuration

To keep the decision reproducible and independent of the protected test set, select between the two Logistic Regression variants using mean cross-validated **F1**. F1 is used here because the task needs a balance between identifying negative reviews and avoiding unnecessary outreach. The Dummy model remains a reference baseline, not a selection candidate.


In [ ]:
logistic_names = ["Logistic Regression", "Logistic Regression (balanced)"]
selected_name = cv_summary.loc[logistic_names, "f1_mean"].idxmax()
selected_pipeline = clone(candidates[selected_name])

print(f"Selected configuration: {selected_name}")
print(f"Mean CV F1: {cv_summary.loc[selected_name, 'f1_mean']:.3f}")


## 3. Threshold selection and final evaluation

### Select the operating threshold using training data

Generate out-of-fold probabilities for the selected Logistic Regression configuration. Choose the threshold that maximises F1 on these training-only predictions. This freezes the operating rule before the protected test set is evaluated.


In [ ]:
oof_probability = cross_val_predict(
    selected_pipeline, X_train, y_train, cv=cv, method="predict_proba"
)[:, 1]

oof_thresholds = threshold_table(y_train, oof_probability)
selected_threshold = best_threshold_by_metric(oof_thresholds, metric="f1")

display(oof_thresholds.round(3))
print(f"Selected threshold (OOF F1): {selected_threshold:.2f}")


### Final protected-test evaluation

Fit the selected pipeline on all training rows, then evaluate it once on the untouched test set. Report precision, recall, F1, balanced accuracy, PR-AUC and ROC-AUC.


In [ ]:
final_pipeline = clone(selected_pipeline).fit(X_train, y_train)
test_probability = final_pipeline.predict_proba(X_test)[:, 1]
test_metrics = classification_metrics(
    y_test, test_probability, threshold=selected_threshold
)

display(pd.Series(test_metrics, name="protected_test").round(3))


## 4. Interpretation

### Logistic Regression coefficients

For the negative-review outcome:
- odds ratio above 1 is associated with higher odds of a negative review;
- odds ratio below 1 is associated with lower odds;
- numeric features are standardised, so their coefficients represent a +1 standard deviation change, holding other model variables fixed.

Coefficients describe **associations**, not causal effects. The table below shows the strongest positive and negative model associations.


In [ ]:
feature_names = final_pipeline.named_steps["preprocessor"].get_feature_names_out()
coefficients = final_pipeline.named_steps["classifier"].coef_[0]

coefficient_table = pd.DataFrame({
    "feature": feature_names,
    "coefficient": coefficients,
})
coefficient_table["odds_ratio"] = np.exp(coefficient_table["coefficient"])
coefficient_table["pct_change_in_odds"] = (
    coefficient_table["odds_ratio"] - 1
) * 100

strongest_associations = pd.concat([
    coefficient_table.nsmallest(10, "coefficient"),
    coefficient_table.nlargest(10, "coefficient"),
]).drop_duplicates()

display(
    strongest_associations
    .sort_values("coefficient")
    .reset_index(drop=True)
    .round({"coefficient": 3, "odds_ratio": 3, "pct_change_in_odds": 1})
)


### Error analysis

Inspect the protected-test errors without using them to retune the model. This gives the team concrete cases to discuss in the report and helps identify limitations.


In [ ]:
test_errors = X_test.copy()
test_errors["actual"] = y_test.to_numpy()
test_errors["probability"] = test_probability
test_errors["predicted"] = (test_probability >= selected_threshold).astype(int)
test_errors["error_type"] = np.select(
    [
        (test_errors["actual"] == 1) & (test_errors["predicted"] == 0),
        (test_errors["actual"] == 0) & (test_errors["predicted"] == 1),
    ],
    ["False negative", "False positive"],
    default="Correct",
)

display(test_errors["error_type"].value_counts().rename("orders").to_frame())
display(
    test_errors.loc[test_errors["error_type"] != "Correct"]
    .sort_values("probability")
    .head(20)
)


## 5. Save the Phase 3 inference artifact

Save the fitted pipeline together with the training-selected threshold, then reload it and score one held-out-shaped row as an interface smoke test.


In [ ]:
ARTIFACT_PATH = PROJECT_ROOT / "deployment" / "negative_review_logistic.joblib"

save_inference_artifact(
    final_pipeline,
    str(ARTIFACT_PATH),
    threshold=selected_threshold,
    metadata={
        "training_data_contract": "Phase 1 order_level.csv + item_level.csv",
        "selection_metric": "cross-validated F1",
        "threshold_metric": "OOF F1",
    },
)

import joblib

artifact = joblib.load(ARTIFACT_PATH)
display(predict_from_artifact(artifact, X_test.iloc[[0]]))
print(f"Saved model artifact: {ARTIFACT_PATH}")
